# AIPAF — AI Project Assessment Framework

### Notebook interattivo · Lorenzo Lombardi

Questo notebook copre l'intero ciclo AIPAF:

| Sezione                        | Contenuto                                                                    |
| ------------------------------ | ---------------------------------------------------------------------------- |
| **0 — Setup**                  | Import dal package installato, provider LLM, RAG opzionale                   |
| **1 — Framework Explorer**     | Visualizzazione config: dimensioni, criteri, pesi per settore                |
| **2 — Assessment interattivo** | Intake + valutazione guidata LLM dimensione per dimensione (con N/A e skip)  |
| **3 — Scoring & Gate**         | Calcolo deterministico score composito, gate (anche PARTIAL), pattern alerts |
| **4 — Report narrativo**       | Executive summary, remediation plan, report Markdown, salvataggio sessione   |
| **5 — Portfolio Analysis**     | Analisi comparativa di sessioni multiple, heatmap, ranking                   |

> **Provider supportati:** Claude (Anthropic API) · Ollama (locale)  
> **Riferimenti normativi:** EU AI Act · NIST AI RMF 1.0 · ISO/IEC 42001:2023  
> **Prerequisito:** package installato nel venv (`uv pip install -e ".[claude,notebook]"`) e kernel `Python (aipaf)`.


---

## Sezione 0 — Setup

Il notebook usa il package `aipaf` installato nel venv: nessuna manipolazione di `sys.path`,
nessuna ricerca del config (è impacchettato in `aipaf/config/`).  
Esegui questa sezione una sola volta per sessione.


In [ ]:
# ── 0.1  Import e configurazione framework ──────────────────────────────────
import os, json, warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets

warnings.filterwarnings("ignore", category=DeprecationWarning)

from dotenv import load_dotenv

load_dotenv(override=False)  # .env nella cartella del notebook: ANTHROPIC_API_KEY, AIPAF_CLAUDE_MODEL, ...

from aipaf import __version__
from aipaf.engine import ScoringEngine, load_config, default_config_path, evaluate_skip_condition
from aipaf.agent import AssessmentAgent, LLMOutputError
from aipaf.llm.factory import get_provider
from aipaf.report import ReportGenerator
from aipaf.models import (
    AssessmentSession,
    ProjectIntake,
    CriterionResponse,
    ConfidenceLevel,
    Sector,
    EUAIActLevel,
    ApprovalGate,
    REMEDIATION_THRESHOLD,
    DIMENSION_CRITICAL_THRESHOLD,
)

cfg = load_config()  # AIPAF_CONFIG_PATH nell'ambiente per un framework personalizzato
engine = ScoringEngine(cfg)

# Cartella di lavoro per sessioni e report (in .gitignore)
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"✓ AIPAF v{__version__}")
print(f"✓ Config: {default_config_path()}")
print(
    f"  framework v{cfg.framework_version or '?'} — {len(cfg.dimensions)} dimensioni, "
    f"{sum(len(d.criteria) for d in cfg.dimensions)} criteri"
)

In [ ]:
# ── 0.2  Provider LLM, RAG opzionale, agent ─────────────────────────────────
LLM_PROVIDER = "ollama"  # "claude" | "ollama"   <── modifica qui
OLLAMA_MODEL = "qwen3.5:4b"  # solo per ollama (vedi README per i modelli consigliati)
USE_RAG = True  # richiede un indice creato con `aipaf rag ingest`

# Claude legge ANTHROPIC_API_KEY (e opzionalmente AIPAF_CLAUDE_MODEL) dall'ambiente.
if LLM_PROVIDER == "claude":
    provider = get_provider("claude")
else:
    provider = get_provider("ollama", model=OLLAMA_MODEL)

if provider.health_check():
    print(f"✓ Provider: {provider.provider_name} / {provider.model_name}")
else:
    print(f"⚠ Provider {provider.provider_name} / {provider.model_name} non raggiungibile")
    print(f"  Motivo: {getattr(provider, 'last_error', None)}")

retriever = None
if USE_RAG:
    try:
        from aipaf.rag.ingestor import DocumentIngestor
        from aipaf.rag.retriever import RAGRetriever

        _ing = DocumentIngestor(embedding_backend=None)  # riusa il backend persistito nell'indice
        retriever = RAGRetriever(_ing)
        _st = _ing.status()
        print(
            f"✓ RAG: {_st['regulatory']['chunks']} chunk normativi, {_st['company']['chunks']} chunk aziendali "
            f"(embedding: {DocumentIngestor.persisted_backend() or 'n/d'})"
        )
    except Exception as e:
        print(f"⚠ RAG disattivato: {e}")

agent = AssessmentAgent(provider=provider, config=cfg, engine=engine, retriever=retriever)
print("✓ Agent pronto")

---

## Sezione 1 — Framework Explorer

Visualizza la struttura del framework: dimensioni, criteri, pesi per settore.  
Utile come riferimento durante l'assessment e per presentazioni.


In [ ]:
# ── 1.1  Panoramica dimensioni ─────────────────────────────────────────────
rows = []
for dim in cfg.dimensions:
    for crit in dim.criteria:
        rows.append(
            {
                "Dim": dim.id,
                "Dimensione": dim.name,
                "Criterio": crit.id,
                "Nome criterio": crit.name,
                "Peso %": crit.weight_pct,
                "Fonte": crit.source,
            }
        )

df_framework = pd.DataFrame(rows)

summary = (
    df_framework.groupby(["Dim", "Dimensione"])
    .agg(Criteri=("Criterio", "count"), Peso_tot=("Peso %", "sum"))
    .reset_index()
)

display(HTML("<h4>Riepilogo dimensioni</h4>"))
display(HTML(summary.to_html(index=False)))

In [ ]:
# ── 1.2  Pesi settoriali ───────────────────────────────────────────────────
sector_rows = []
for sector_name, sw in cfg.sector_weights.items():
    sector_rows.append(
        {
            "Settore": sector_name,
            "D1": sw.D1,
            "D2": sw.D2,
            "D3": sw.D3,
            "D4": sw.D4,
            "D5": sw.D5,
            "D6": sw.D6,
        }
    )

df_sectors = pd.DataFrame(sector_rows).set_index("Settore")

fig, ax = plt.subplots(figsize=(11, 4))
df_sectors.plot(kind="bar", ax=ax, colormap="tab20", width=0.75, edgecolor="white")
ax.set_title("Pesi dimensionali per settore (%)", fontsize=13, pad=12)
ax.set_ylabel("Peso %")
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(title="Dimensione", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.axhline(100 / 6, ls="--", color="gray", alpha=0.4, label="Media uniforme")
plt.tight_layout()
plt.show()

In [ ]:
# ── 1.3  Dettaglio criteri con scoring guide ───────────────────────────────
dim_selector = widgets.Dropdown(
    options=[(f"{d.id} — {d.name}", d.id) for d in cfg.dimensions],
    description="Dimensione:",
    layout=widgets.Layout(width="420px"),
)

out_criteria = widgets.Output()


def show_criteria(change):
    with out_criteria:
        clear_output(wait=True)
        dim = cfg.dimension_map[dim_selector.value]
        print(f"\n{'=' * 64}")
        print(f"  {dim.id} — {dim.name}")
        print(f"  {dim.description}")
        print(f"  Compilato da: {dim.compiled_by}")
        print(f"{'=' * 64}")
        for c in dim.criteria:
            print(f"\n  [{c.id}]  {c.name}  (peso {c.weight_pct}%)")
            print(f"  {c.description}")
            print(f"  Fonte: {c.source}")
            print(f"  Scale:  3 → {c.scoring_guide.s3[:80]}")
            print(f"          2 → {c.scoring_guide.s2[:80]}")
            print(f"          1 → {c.scoring_guide.s1[:80]}")
            print(f"          0 → {c.scoring_guide.s0[:80]}")


dim_selector.observe(show_criteria, names="value")
show_criteria(None)
display(dim_selector, out_criteria)

---

## Sezione 2 — Assessment interattivo

### 2.1 — Intake progetto

Inserisci la descrizione del progetto. L'LLM estrae automaticamente
i metadati strutturati (settore, EU AI Act level, tipo dati trattati, ecc.).


In [ ]:
# ── 2.1  Intake da testo libero ────────────────────────────────────────────
PROJECT_DESCRIPTION = """
Inserisci qui la descrizione del progetto AI da valutare.

Includere:
- Obiettivo del progetto e use case principale
- Settore (banking, healthcare, manufacturing, public_admin, media_telecom...)
- Stadio (PoC / Pilota / Produzione)
- Tipo di sistema AI (classificatore, LLM, recommender, visione, ecc.)
- Tipologia di dati trattati (personali, sensibili, biometrici...) — o l'assenza di dati personali
- Utenti finali e contesto di utilizzo
- Eventuali soggetti vulnerabili coinvolti
"""

session = AssessmentSession()
ASSESSED_DIMS = set()  # dimensioni valutate in questa sessione (per il gate PARTIAL)

print("Analisi progetto in corso con", provider.provider_name, "...")
try:
    session = agent.collect_intake(session, PROJECT_DESCRIPTION)
except LLMOutputError as e:
    raise SystemExit(
        f"Intake non interpretabile dall'LLM: {e}\n"
        "Senza settore e livello EU AI Act validi l'assessment non può procedere."
    )
intake = session.intake

print("\n" + "=" * 56)
print(f"  Progetto  : {intake.project_name}")
print(f"  Use case  : {intake.use_case_description}")
print(f"  Owner     : {intake.project_owner}")
print(f"  Settore   : {intake.sector.value}")
print(f"  Stadio    : {intake.project_stage}")
print(f"  AI type   : {intake.ai_system_type}")
print(f"  EU AI Act : {intake.eu_ai_act_level.value}")
print(f"  Utenti    : {intake.user_count_estimate if intake.user_count_estimate is not None else 'n/d'}")
print(f"  Dati pers.: {'Sì' if intake.personal_data_treated else 'No'}")
print(f"  Dati sens.: {'Sì' if intake.sensitive_data_treated else 'No'}")
print(f"  Vulnerabili: {'Sì' if intake.vulnerable_subjects else 'No'}")
print("=" * 56)

AUTO_NA = engine.candidate_not_applicable(intake)
if AUTO_NA:
    print(f"\nℹ Criteri non applicabili in base all'intake (skip_condition): {', '.join(AUTO_NA)}")
    print("  Verranno marcati N/A se lasciati vuoti nelle celle seguenti.")

---

### 2.2 — Valutazione criteri

Per ogni criterio: descrivi la situazione attuale con le evidenze disponibili.  
L'LLM interpreta la risposta e assegna score (0–3) e confidence (H/M/L).

**Regola confidence:**

- **H** — evidenza documentata, metriche verificabili, output di tool
- **M** — auto-dichiarazione senza documentazione formale
- **L** — stima o dato assente → score azzerato (`data_needed`)

**Valori speciali** nel dizionario delle risposte:

- `""` (vuoto) → criterio **saltato**: pesa come L (score 0, _data needed_). Il criterio si applica ma non è stato valutato.
- `"na"` oppure `"na: motivazione"` → **non applicabile**: esce dal denominatore della dimensione, non pesa.
- chiave assente e criterio in `AUTO_NA` → N/A automatico dalla `skip_condition` (es. D2.C2 senza dati personali).

Esegui solo le dimensioni che vuoi valutare: quelle non eseguite restano fuori da composito e gate,
e il gate risulta **PARTIAL**. Puoi rieseguire una cella per aggiornare i criteri di quella dimensione.


In [ ]:
# ── 2.2  Funzione di valutazione (usata dalle celle D1–D6) ──────────────────
_SCORE_ICONS = {3: "🟢", 2: "🔵", 1: "🟡", 0: "🔴"}


def assess_dimension(dim_id: str, answers: dict) -> None:
    dim_cfg = cfg.dimension_map[dim_id]
    print("\n" + "=" * 56)
    print(f"  {dim_id} — {dim_cfg.name}   ({len(dim_cfg.criteria)} criteri)")
    print("=" * 56)
    failed = []
    for crit in dim_cfg.criteria:
        answer = (answers.get(crit.id) or "").strip()
        token = answer.lower()

        if token in ("na", "n/a") or token.startswith("na:"):
            reason = answer.split(":", 1)[1].strip() if ":" in answer else "Non applicabile al progetto"
            resp = CriterionResponse(criterion_id=crit.id, not_applicable=True, evidence=reason)
            print(f"  [{crit.id}]  ⊘ N/A — {reason}")

        elif not answer and crit.id in AUTO_NA:
            resp = CriterionResponse(
                criterion_id=crit.id, not_applicable=True, evidence=f"Non applicabile: {crit.skip_condition}"
            )
            print(f"  [{crit.id}]  ⊘ N/A automatico ({crit.skip_condition})")

        elif not answer:
            resp = CriterionResponse(
                criterion_id=crit.id, score=0, confidence=ConfidenceLevel.LOW, evidence="Criterio non compilato"
            )
            print(f"  [{crit.id}]  ⏭  saltato (score=0, L, data needed)")

        else:
            try:
                resp = agent.interpret_criterion_response(crit.id, answer, dim_cfg)
                print(
                    f"  [{crit.id}]  {_SCORE_ICONS.get(resp.score, '')} score={resp.score}/3 "
                    f"conf={resp.confidence.value} eff={resp.effective_score:.1f}"
                )
                if resp.notes:
                    print(f"         {resp.notes[:100]}")
            except Exception as e:  # LLMOutputError, rete, timeout
                failed.append(crit.id)
                resp = CriterionResponse(
                    criterion_id=crit.id,
                    score=0,
                    confidence=ConfidenceLevel.LOW,
                    evidence=answer[:200],
                    notes=f"INTERPRETAZIONE LLM FALLITA: {e}",
                )
                print(f"  [{crit.id}]  ⚠ interpretazione fallita → registrato L (score 0): {str(e)[:80]}")
        session.upsert_response(resp)

    ASSESSED_DIMS.add(dim_id)
    print(f"\n✓ {dim_id} completata" + (f"  ⚠ da rivedere: {', '.join(failed)}" if failed else ""))

#### D1 — Strategic Alignment & Business Value


In [ ]:
D1_ANSWERS = {
    # "D1.C1": "descrizione della situazione, con le evidenze disponibili...",
}
assess_dimension("D1", D1_ANSWERS)

#### D2 — Risk Classification & Regulatory Compliance


In [ ]:
D2_ANSWERS = {
    # "D2.C1": "descrizione della situazione, con le evidenze disponibili...",
    # "D2.C2": "na: il sistema non tratta dati personali",
}
assess_dimension("D2", D2_ANSWERS)

#### D3 — Data Readiness & Governance


In [ ]:
D3_ANSWERS = {
    # "D3.C1": "descrizione della situazione, con le evidenze disponibili...",
}
assess_dimension("D3", D3_ANSWERS)

#### D4 — Technical Feasibility & Architecture


In [ ]:
D4_ANSWERS = {
    # "D4.C1": "descrizione della situazione, con le evidenze disponibili...",
}
assess_dimension("D4", D4_ANSWERS)

#### D5 — Ethical & Responsible AI


In [ ]:
D5_ANSWERS = {
    # "D5.C1": "descrizione della situazione, con le evidenze disponibili...",
}
assess_dimension("D5", D5_ANSWERS)

#### D6 — Operational Readiness & Lifecycle Management


In [ ]:
D6_ANSWERS = {
    # "D6.C1": "descrizione della situazione, con le evidenze disponibili...",
}
assess_dimension("D6", D6_ANSWERS)

---

## Sezione 3 — Scoring & Gate Decision

Calcolo deterministico dello score composito, gate, pattern alerts e flag normativi.  
Riesegui questa sezione ogni volta che aggiorni le risposte nella Sezione 2.  
Le dimensioni non eseguite in Sezione 2 restano escluse: il gate è **PARTIAL** finché non sono tutte valutate.


In [ ]:
# ── 3.1  Calcolo score ─────────────────────────────────────────────────────
session = agent.finalize(session, assessed_dimensions=sorted(ASSESSED_DIMS))
print(engine.summary(session))
if session.partial:
    print(
        f"\n⚠ Assessment PARZIALE — non valutate: {', '.join(session.unassessed_dimensions)}. Il gate non è definitivo."
    )

In [ ]:
# ── 3.2  Radar chart dimensioni ────────────────────────────────────────────
dims = session.dimension_scores
labels = [d.dimension_id for d in dims]
# Le dimensioni non valutate o N/A non hanno uno score: nel radar restano a 0 con etichetta
values = [d.raw_score if d.counts else 0.0 for d in dims]
tags = ["" if d.counts else ("n/v" if not d.assessed else "N/A") for d in dims]

N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values_plot = values + [values[0]]
angles_plot = angles + [angles[0]]

gate_colors = {
    ApprovalGate.GREEN: ("#27ae60", "#2ecc71"),
    ApprovalGate.YELLOW: ("#f39c12", "#f1c40f"),
    ApprovalGate.RED: ("#c0392b", "#e74c3c"),
    ApprovalGate.PARTIAL: ("#7f8c8d", "#bdc3c7"),
}
edge_col, fill_col = gate_colors.get(session.gate, ("#3498db", "#5dade2"))

fig = plt.figure(figsize=(14, 5.5))
ax = fig.add_subplot(121, polar=True)
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 3)
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["1", "2", "3"], fontsize=8, color="gray")
ax.plot(angles_plot, values_plot, color=edge_col, linewidth=2.5)
ax.fill(angles_plot, values_plot, color=fill_col, alpha=0.28)
for a, v, t in zip(angles, values, tags):
    ax.annotate(
        t or f"{v:.2f}",
        xy=(a, v),
        xytext=(a, v + 0.18),
        fontsize=9,
        ha="center",
        fontweight="bold",
        color=edge_col if not t else "gray",
    )
gate_val = session.gate.value if session.gate else "N/A"
if session.partial and session.provisional_gate:
    gate_val += f" (provv. {session.provisional_gate.value})"
ax.set_title(
    f"Score per dimensione\nGate: {gate_val}  |  Composito: {session.composite_score:.2f}/3.00", pad=18, fontsize=12
)

ax2 = fig.add_subplot(122)
bars = ax2.barh(
    [f"{d.dimension_id}  ({d.weight_pct}%)" for d in dims],
    values,
    color=[
        "#bdc3c7" if t else "#27ae60" if v >= 2 else "#f39c12" if v >= 1 else "#e74c3c" for v, t in zip(values, tags)
    ],
    edgecolor="white",
    height=0.6,
)
ax2.set_xlim(0, 3.3)
ax2.axvline(2.0, ls="--", color="#27ae60", alpha=0.7, lw=1.2, label="Soglia adeguato (2.0)")
ax2.axvline(1.0, ls="--", color="#e74c3c", alpha=0.7, lw=1.2, label="Soglia critico (1.0)")
ax2.axvline(
    DIMENSION_CRITICAL_THRESHOLD,
    ls=":",
    color="#c0392b",
    alpha=0.7,
    lw=1.2,
    label=f"RED forzato (< {DIMENSION_CRITICAL_THRESHOLD})",
)
for bar, v, t in zip(bars, values, tags):
    ax2.text(v + 0.05, bar.get_y() + bar.get_height() / 2, t or f"{v:.2f}", va="center", fontsize=10, fontweight="bold")
ax2.set_xlabel("Score (0–3)", fontsize=10)
ax2.set_title("Score per dimensione (con pesi settoriali)", fontsize=12)
ax2.legend(fontsize=9)
ax2.invert_yaxis()

plt.suptitle(
    f"AIPAF — {session.intake.project_name if session.intake else 'Assessment'}", fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3  Pattern Alerts e Flag Normativi ───────────────────────────────────
active_patterns = [p for p in session.pattern_alerts if p.triggered]
if active_patterns:
    display(HTML("<h4>⚡ Pattern Alert attivi</h4>"))
    for p in active_patterns:
        display(
            HTML(f"""
        <div style='background:#fff3cd;border-left:4px solid #f39c12;
                    padding:10px 14px;margin:6px 0;border-radius:4px;'>
          <b>{p.label}</b> — {p.name}<br>
          <span style='color:#666'>{p.description}</span><br>
          <span style='color:#e67e22'>⟶ {p.recommended_action}</span>
        </div>""")
        )
else:
    display(HTML("<p style='color:green'>✓ Nessun pattern critico rilevato</p>"))

if session.normative_red_flags:
    display(HTML("<h4>🚨 Flag Normativi</h4>"))
    for f in session.normative_red_flags:
        display(
            HTML(f"""
        <div style='background:#fdecea;border-left:4px solid #e74c3c;
                    padding:10px 14px;margin:6px 0;border-radius:4px;'>
          <b>{f.norm_reference}</b> — [{f.criterion_id}] {f.criterion_name}<br>
          <span style='color:#666'>{f.description}</span>
        </div>""")
        )
else:
    display(HTML("<p style='color:green'>✓ Nessun flag normativo</p>"))

---

## Sezione 4 — Report Narrativo e salvataggio

Executive summary e piano di remediation generati dall'LLM (con contesto RAG se attivo),
poi report Markdown e sessione JSON in `output/`.  
Richiede il provider LLM attivo per 4.1 e 4.2; 4.3 funziona anche senza.


In [ ]:
# ── 4.1  Executive Summary ─────────────────────────────────────────────────
print("Generazione executive summary in corso...\n")
exec_summary = agent.generate_executive_summary(session)
display(Markdown("### Executive Summary\n\n" + exec_summary))

In [ ]:
# ── 4.2  Remediation Plan ──────────────────────────────────────────────────
weak_dims = [d for d in session.dimension_scores if d.counts and d.raw_score < REMEDIATION_THRESHOLD]
if weak_dims:
    print(f"Dimensioni sotto soglia: {[d.dimension_id for d in weak_dims]}")
    print("Generazione remediation plan...\n")
    remediation = agent.generate_remediation_plan(session)
    display(Markdown("### Remediation Plan\n\n" + remediation))
else:
    remediation = ""
    display(
        HTML(
            f"<p style='color:green'>✓ Tutte le dimensioni valutate ≥ {REMEDIATION_THRESHOLD} — nessun remediation plan necessario</p>"
        )
    )

In [ ]:
# ── 4.3  Report Markdown e salvataggio sessione ────────────────────────────
project_slug = session.intake.project_name.lower().replace(" ", "_") if session.intake else "assessment"
stamp = date.today().isoformat()

session_path = OUTPUT_DIR / f"assessment_{project_slug}_{stamp}.json"
with open(session_path, "w", encoding="utf-8") as f:
    json.dump(session.model_dump(mode="json"), f, indent=2, ensure_ascii=False)
print(f"✓ Sessione salvata: {session_path}")
print(
    f"  LLM: {session.llm_provider}/{session.llm_model} · AIPAF v{session.aipaf_version} · "
    f"copertura: {'parziale' if session.partial else 'completa'}"
)

report_path = ReportGenerator().generate_and_save(
    session,
    executive_summary=globals().get("exec_summary", ""),
    remediation_plan=globals().get("remediation", ""),
    output_path=OUTPUT_DIR / f"AIPAF_Report_{project_slug}_{stamp}.md",
)
print(f"✓ Report Markdown: {report_path}")

---

## Sezione 5 — Portfolio Analysis

Carica e confronta più sessioni di assessment salvate.  
Utile per monitorare l'evoluzione di un progetto nel tempo  
o comparare più progetti in un portfolio.


In [ ]:
# ── 5.1  Caricamento sessioni ──────────────────────────────────────────────
# Sessioni salvate da 4.3 (output/) o dalla CLI (--output) nella cartella corrente
session_files = sorted(set(list(OUTPUT_DIR.glob("assessment_*.json")) + list(Path(".").glob("assessment_*.json"))))
print(f"Trovate {len(session_files)} sessioni:")
for f in session_files:
    print(f"  {f}")

sessions: list[AssessmentSession] = []
for f in session_files:
    try:
        with open(f, encoding="utf-8") as fh:
            s = AssessmentSession.model_validate(json.load(fh))
        if s.composite_score is None:
            s = engine.compute(s)  # sessioni 0.1.x: ricalcolo con le regole correnti
        sessions.append(s)
    except Exception as e:
        print(f"  ⚠ Errore caricamento {f.name}: {e}")

print(f"\n✓ {len(sessions)} sessioni caricate")

In [ ]:
# ── 5.2  Tabella comparativa ────────────────────────────────────────────────
if not sessions:
    print("Nessuna sessione disponibile — esegui prima un assessment (Sezioni 2-4)")
else:
    portfolio_rows = []
    for s in sessions:
        row = {
            "Progetto": s.intake.project_name if s.intake else "N/A",
            "Settore": s.intake.sector.value if s.intake else "",
            "EU AI Act": s.intake.eu_ai_act_level.value if s.intake else "",
            "LLM": f"{s.llm_provider}/{s.llm_model}" if s.llm_provider else "",
            "Score": round(s.composite_score or 0, 2),
            "Gate": s.gate.value if s.gate else "N/A",
        }
        for d in s.dimension_scores:
            row[d.dimension_id] = round(d.raw_score, 2) if d.counts else np.nan  # n/v o N/A → vuoto
        portfolio_rows.append(row)

    df_portfolio = pd.DataFrame(portfolio_rows)

    def _row_color(row):
        colors = []
        for col, val in row.items():
            if col == "Gate":
                c = {
                    "GREEN": "background-color:#d4edda",
                    "YELLOW": "background-color:#fff3cd",
                    "RED": "background-color:#f8d7da",
                    "PARTIAL": "background-color:#e9ecef",
                }.get(val, "")
            elif isinstance(val, float) and not np.isnan(val):
                c = (
                    "background-color:#d4edda"
                    if val >= 2
                    else "background-color:#fff3cd"
                    if val >= 1
                    else "background-color:#f8d7da"
                )
            else:
                c = ""
            colors.append(c)
        return colors

    try:
        display(
            df_portfolio.style.apply(_row_color, axis=1)
            .format({c: "{:.2f}" for c in df_portfolio.select_dtypes("float").columns}, na_rep="—")
            .set_caption("Portfolio Assessment AIPAF")
            .hide(axis="index")
        )
    except Exception:
        display(HTML(df_portfolio.to_html(index=False, na_rep="—")))

In [ ]:
# ── 5.3  Heatmap portfolio ─────────────────────────────────────────────────
if len(sessions) >= 2:
    import seaborn as sns

    dim_cols = [c for c in df_portfolio.columns if c in cfg.dimension_ids]
    heat_data = df_portfolio.set_index("Progetto")[dim_cols]

    fig, ax = plt.subplots(figsize=(max(8, len(dim_cols) * 1.4), max(4, len(sessions) * 0.9)))
    sns.heatmap(
        heat_data,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0,
        vmax=3,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
        cbar_kws={"label": "Score (0–3)"},
    )
    ax.set_title("Heatmap score per dimensione — Portfolio AIPAF", pad=12, fontsize=12)
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()
elif sessions:
    print("Carica almeno 2 sessioni per la heatmap portfolio")
else:
    print("Nessuna sessione disponibile")

In [ ]:
# ── 5.4  Trend nel tempo (stesso progetto, versioni diverse) ────────────────
# Filtra sessioni con lo stesso nome progetto, ordinate per data
if sessions:
    project_names = df_portfolio["Progetto"].unique().tolist()
    target_project = project_names[0]  # <── modifica per selezionare un progetto specifico

    trend_sessions = sorted(
        [s for s in sessions if s.intake and s.intake.project_name == target_project],
        key=lambda s: s.intake.assessment_date if s.intake else "",
    )

    if len(trend_sessions) >= 2:
        fig, ax = plt.subplots(figsize=(10, 5))
        dim_ids = [d.dimension_id for d in trend_sessions[0].dimension_scores]
        dates = [s.intake.assessment_date for s in trend_sessions]

        for dim_id in dim_ids:
            scores_over_time = []
            for s in trend_sessions:
                dim_s = next((d for d in s.dimension_scores if d.dimension_id == dim_id), None)
                scores_over_time.append(dim_s.raw_score if (dim_s and dim_s.counts) else np.nan)
            ax.plot(dates, scores_over_time, marker="o", label=dim_id, linewidth=2)

        ax.set_ylim(0, 3.2)
        ax.axhline(2.0, ls="--", color="green", alpha=0.4)
        ax.axhline(1.0, ls="--", color="red", alpha=0.4)
        ax.set_title(f"Trend assessment — {target_project}", fontsize=12)
        ax.set_xlabel("Data assessment")
        ax.set_ylabel("Score (0–3)")
        ax.legend(title="Dimensione", bbox_to_anchor=(1.01, 1), loc="upper left")
        plt.xticks(rotation=25)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Solo {len(trend_sessions)} sessione trovata per '{target_project}' — servono >= 2 per il trend")

---
## Note e riferimenti

### Scala di valutazione
| Score | Label | Significato |
|-------|-------|-------------|
| 3 | Excellent | Best practice, evidenze documentate |
| 2 | Adequate | Gap identificati, remediation plan definito |
| 1 | Insufficient | Gap significativi, intervento necessario |
| 0 | Critical | Non soddisfatto, stop obbligatorio |

### Confidence e effective score
| Confidence | Moltiplicatore | Quando |
|------------|----------------|--------|
| H (High) | ×1.0 | Evidenza documentata e verificabile |
| M (Medium) | ×0.8 | Auto-dichiarazione senza prove formali |
| L (Low) | ×0.0 | Stima o assenza di dato → `data_needed` |

### Criteri saltati e non applicabili
| Caso | Effetto sullo score |
|------|---------------------|
| Saltato (`""`) | Confidence L: score 0, **pesa** sulla dimensione, flag *data needed* |
| N/A (`"na"` o `skip_condition`) | **Escluso dal denominatore**: la dimensione è la media pesata dei soli criteri applicabili |
| Dimensione con tutti i criteri N/A | Esclusa da composito e gate, con nota esplicita |

### Gate
| Gate | Composito | Condizione aggiuntiva |
|------|-----------|----------------------|
| 🟢 GREEN | ≥ 2.0 | Nessuna dimensione < 0.5 |
| 🟡 YELLOW | 1.0–1.9 | — |
| 🔴 RED | < 1.0 | **oppure** qualsiasi dimensione < 0.5 |
| ⚪ PARTIAL | — | Almeno una dimensione non valutata: `provisional_gate` sulle sole dimensioni valutate, nessuna decisione definitiva |

### Riferimenti normativi
- EU AI Act — Regolamento (UE) 2024/1689
- NIST AI RMF 1.0 — AI Risk Management Framework
- ISO/IEC 42001:2023 — AI Management System
- IBM — Guide for Implementing an AI Governance Framework (2025)

---

_AIPAF — AI Project Assessment Framework · Lorenzo Lombardi_
